# Cell 1 -- Installing Required Libraries

Before we begin, we need to install a few external Python libraries that are not available by default in the Kaggle environment.

- **transformers**: Provides access to pretrained deep learning models like HuBERT, which is a pretrained audio Transformer used for audio classification tasks.
- **wandb** (Weights and Biases): A tool for experiment tracking. It logs all our training metrics (loss, accuracy, F1 score, learning rate) so we can compare models easily on a dashboard.
- **librosa**: A popular Python library for audio analysis. We use it to extract handcrafted audio features like MFCCs, Chroma, and Spectral Contrast for our XGBoost baseline.
- **xgboost**: A powerful gradient-boosted tree algorithm used as our traditional ML baseline model.

The `-q` flag keeps the installation output quiet.


# Cell 1 -- Installing Required Libraries

Before we begin, we need to install a few external Python libraries that are not available by default in the Kaggle environment.

- **transformers**: Provides access to pretrained deep learning models like HuBERT, which is a pretrained audio Transformer used for audio classification tasks.
- **wandb** (Weights and Biases): A tool for experiment tracking. It logs all our training metrics (loss, accuracy, F1 score, learning rate) so we can compare models easily on a dashboard.
- **librosa**: A popular Python library for audio analysis. We use it to extract handcrafted audio features like MFCCs, Chroma, and Spectral Contrast for our XGBoost baseline.
- **xgboost**: A powerful gradient-boosted tree algorithm used as our traditional ML baseline model.

The `-q` flag keeps the installation output quiet.


In [1]:
# ============================================================
# Cell 1: Installs (Run on CPU - no GPU needed)
# ============================================================
!pip install transformers wandb librosa xgboost -q

# Cell 2 -- Imports, Seed, and Configuration

This cell does three important things:

**1. Imports:** We import all the libraries we will use throughout this notebook, including PyTorch (for building neural networks), torchaudio (for audio loading and transformations), scikit-learn (for evaluation metrics), and others.

**2. Reproducibility (Seed):** We set `SEED = 42` and apply it to Python's `random`, NumPy, and PyTorch. This ensures that every time we run the notebook, the random operations (like shuffling data or initializing weights) produce the same results, making our experiments reproducible.

**3. Constants:**
- `SAMPLE_RATE = 16000`: All audio is resampled to 16kHz. This is the standard rate for speech/audio models like HuBERT.
- `DURATION = 10`: We process 10-second audio clips. Test mashups are 30 seconds long, but our models are trained on 10-second segments.
- `NUM_CLASSES = 10`: We have 10 music genres to classify: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock.
- `genre_to_idx` / `idx_to_genre`: Dictionaries to convert between genre names and numeric labels (required by PyTorch).


# Cell 2 -- Imports, Seed, and Configuration

This cell does three important things:

**1. Imports:** We import all the libraries we will use throughout this notebook, including PyTorch (for building neural networks), torchaudio (for audio loading and transformations), scikit-learn (for evaluation metrics), and others.

**2. Reproducibility (Seed):** We set `SEED = 42` and apply it to Python's `random`, NumPy, and PyTorch. This ensures that every time we run the notebook, the random operations (like shuffling data or initializing weights) produce the same results, making our experiments reproducible.

**3. Constants:**
- `SAMPLE_RATE = 16000`: All audio is resampled to 16kHz. This is the standard rate for speech/audio models like HuBERT.
- `DURATION = 10`: We process 10-second audio clips. Test mashups are 30 seconds long, but our models are trained on 10-second segments.
- `NUM_CLASSES = 10`: We have 10 music genres to classify: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock.
- `genre_to_idx` / `idx_to_genre`: Dictionaries to convert between genre names and numeric labels (required by PyTorch).


In [2]:
# ============================================================
# Cell 2: Imports + Seed + Config
# ============================================================
import os
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split

import librosa
import librosa.display
import IPython.display as ipd

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report

import wandb
from transformers import HubertForSequenceClassification
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Constants
SAMPLE_RATE = 16000
DURATION = 10         # seconds (increased from 5 for better genre recognition)
NUM_SAMPLES = SAMPLE_RATE * DURATION
NUM_CLASSES = 10

genres = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
genre_to_idx = {g: i for i, g in enumerate(genres)}
idx_to_genre = {i: g for i, g in enumerate(genres)}

print("All imports done.")
print(f"PyTorch: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

All imports done.
PyTorch: 2.8.0+cu126
GPU Available: True
GPU Name: Tesla T4


# Cell 3 -- Path Configuration and Directory Exploration

Here we set up all the file paths for our dataset and explore its structure.

**Key directories:**
- `STEMS_DIR`: Contains the training data. Each genre has 100 songs, and each song has 4 stems: `vocals.wav`, `drums.wav`, `bass.wav`, `other.wav`.
- `MASHUPS_DIR`: Contains 3020 test audio files (mashups). These are noisy mixes of stems from different songs of the same genre, with added environmental noise.
- `NOISE_DIR`: Points to the ESC-50 dataset (2000 environmental sound clips). We use this to inject realistic background noise into our training data to make our model robust to the noisy test conditions.

**Important:** The test mashups are created by mixing stems from *different* songs of the same genre, not from the same song. This insight is critical and it motivates our Cross-Song Stem Mixing augmentation strategy in later cells.


# Cell 3 -- Path Configuration and Directory Exploration

Here we set up all the file paths for our dataset and explore its structure.

**Key directories:**
- `STEMS_DIR`: Contains the training data. Each genre has 100 songs, and each song has 4 stems: `vocals.wav`, `drums.wav`, `bass.wav`, `other.wav`.
- `MASHUPS_DIR`: Contains 3020 test audio files (mashups). These are noisy mixes of stems from different songs of the same genre, with added environmental noise.
- `NOISE_DIR`: Points to the ESC-50 dataset (2000 environmental sound clips). We use this to inject realistic background noise into our training data to make our model robust to the noisy test conditions.

**Important:** The test mashups are created by mixing stems from *different* songs of the same genre, not from the same song. This insight is critical and it motivates our Cross-Song Stem Mixing augmentation strategy in later cells.


In [3]:
# ============================================================
# Cell 3: Path Config + Directory Exploration
# CRITICAL: This cell discovers the actual file naming format
# ============================================================
BASE_DIR = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'

# Fallback for older competition path structure
if not os.path.exists(BASE_DIR):
    BASE_DIR = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'

STEMS_DIR = os.path.join(BASE_DIR, 'genres_stems')
MASHUPS_DIR = os.path.join(BASE_DIR, 'mashups')

# Auto-detect noise folder (ESC-50-master vs ESC-50)
if os.path.exists(os.path.join(BASE_DIR, 'ESC-50-master', 'audio')):
    NOISE_DIR = os.path.join(BASE_DIR, 'ESC-50-master', 'audio')
elif os.path.exists(os.path.join(BASE_DIR, 'ESC-50', 'audio')):
    NOISE_DIR = os.path.join(BASE_DIR, 'ESC-50', 'audio')
else:
    NOISE_DIR = None
    print("WARNING: Noise directory not found!")

print(f"BASE_DIR:   {BASE_DIR}")
print(f"STEMS_DIR:  {STEMS_DIR}")
print(f"MASHUPS_DIR:{MASHUPS_DIR}")
print(f"NOISE_DIR:  {NOISE_DIR}")
print()

# --- Explore stems directory ---
print("=" * 50)
print("TRAINING DATA (Stems per Genre)")
print("=" * 50)
for genre in genres:
    genre_path = os.path.join(STEMS_DIR, genre)
    if os.path.exists(genre_path):
        songs = sorted(os.listdir(genre_path))
        # Check what stems exist in first song
        first_song_stems = os.listdir(os.path.join(genre_path, songs[0])) if songs else []
        print(f"  {genre:12s}: {len(songs)} songs | stems: {first_song_stems}")

# --- CRITICAL: Explore mashup files to discover naming format ---
print()
print("=" * 50)
print("TEST DATA (Mashup Files)")
print("=" * 50)
mashup_files = sorted(os.listdir(MASHUPS_DIR))
print(f"Total mashup files: {len(mashup_files)}")
print(f"First 10 files: {mashup_files[:10]}")
print(f"Last 5 files:   {mashup_files[-5:]}")

# --- Check test.csv ID format ---
print()
print("=" * 50)
print("test.csv and sample_submission.csv")
print("=" * 50)
test_csv_path = os.path.join(BASE_DIR, 'test.csv')
sample_sub_path = os.path.join(BASE_DIR, 'sample_submission.csv')

test_df = pd.read_csv(test_csv_path)
print(f"test.csv shape: {test_df.shape}")
print(f"test.csv columns: {list(test_df.columns)}")
print(f"First 5 rows:\n{test_df.head()}")
print(f"\nID dtype: {test_df['id'].dtype}")
print(f"Sample IDs: {test_df['id'].values[:10]}")

if os.path.exists(sample_sub_path):
    sample_sub = pd.read_csv(sample_sub_path)
    print(f"\nsample_submission.csv:\n{sample_sub.head()}")

# --- Noise files count ---
if NOISE_DIR:
    noise_files = [f for f in os.listdir(NOISE_DIR) if f.endswith('.wav')]
    print(f"\nNoise files (ESC-50): {len(noise_files)}")

BASE_DIR:   /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup
STEMS_DIR:  /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems
MASHUPS_DIR:/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups
NOISE_DIR:  /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio

TRAINING DATA (Stems per Genre)
  blues       : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  classical   : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  country     : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  disco       : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  hiphop      : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  jazz        : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  metal       : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  pop         : 100 songs | stems: ['drums.wav

# Cell 9 -- GPU Device Setup

This cell sets up the GPU device for PyTorch. If a CUDA GPU is available (like Tesla T4 on Kaggle), we use it. Otherwise, we fall back to CPU. GPUs massively speed up neural network training due to their parallel processing capabilities.


# Cell 9 -- GPU Device Setup

This cell sets up the GPU device for PyTorch. If a CUDA GPU is available (like Tesla T4 on Kaggle), we use it. Otherwise, we fall back to CPU. GPUs massively speed up neural network training due to their parallel processing capabilities.


In [9]:
# ============================================================
# Cell 9: XGBoost Test Submission
# ============================================================

def extract_features_from_file(file_path, sr=SAMPLE_RATE, duration=DURATION):
    """Extract features from a single audio file."""
    y, _ = librosa.load(file_path, sr=sr, duration=duration)
    num_frames = int(sr * duration)
    if len(y) < num_frames:
        y = np.pad(y, (0, num_frames - len(y)))
    else:
        y = y[:num_frames]
    max_val = np.max(np.abs(y))
    if max_val > 0:
        y = y / max_val
    return extract_features_from_array(y, sr)


# Build file lookup for mashups (handles any naming convention)
mashup_file_lookup = {}
for f in os.listdir(MASHUPS_DIR):
    full_path = os.path.join(MASHUPS_DIR, f)
    mashup_file_lookup[f] = full_path
    name_no_ext = os.path.splitext(f)[0]
    mashup_file_lookup[name_no_ext] = full_path

print("Extracting features from test mashups...")
test_df = pd.read_csv(os.path.join(BASE_DIR, 'test.csv'))
X_test_ml = []
test_ids_ml = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test features"):
    file_id = str(row['id'])

    # Try multiple naming conventions (actual format: song0001.wav)
    file_path = None
    candidates = [
        f"song{int(file_id):04d}.wav",  # song0001.wav
        f"{file_id}.wav",
        file_id,
    ]
    # Also try using the 'filename' column if it exists
    if 'filename' in row and pd.notna(row['filename']):
        fname = os.path.basename(str(row['filename']))
        candidates.insert(0, fname)

    for c in candidates:
        if c in mashup_file_lookup:
            file_path = mashup_file_lookup[c]
            break

    if file_path is None:
        print(f"WARNING: File not found for id={file_id}, using zeros")
        X_test_ml.append(np.zeros(X_train_ml.shape[1]))
    else:
        feat = extract_features_from_file(file_path)
        X_test_ml.append(feat)
    test_ids_ml.append(int(file_id))

X_test_ml = np.array(X_test_ml)

# Predict
y_test_pred = clf.predict(X_test_ml)
y_test_genres = le.inverse_transform(y_test_pred)

# Save submission
sub_xgb = pd.DataFrame({'id': test_ids_ml, 'genre': y_test_genres})
sub_xgb.to_csv('submission_xgb.csv', index=False)
print(f"\nXGBoost submission saved: submission_xgb.csv")
print(sub_xgb.head(10))
print(f"\nGenre distribution in predictions:")
print(sub_xgb['genre'].value_counts())

Extracting features from test mashups...


Test features:   6%|▌         | 167/3020 [00:22<06:35,  7.21it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
Test features: 100%|██████████| 3020/3020 [07:00<00:00,  7.18it/s]



XGBoost submission saved: submission_xgb.csv
   id      genre
0   1        pop
1   2  classical
2   3        pop
3   4     hiphop
4   5    country
5   6     hiphop
6   7     hiphop
7   8     hiphop
8   9        pop
9  10     reggae

Genre distribution in predictions:
genre
reggae       1193
hiphop        694
pop           422
disco         188
blues         186
country       104
classical      91
jazz           66
metal          44
rock           32
Name: count, dtype: int64


## Milestone 3-5: Deep Learning Models (CNN, CRNN, HuBERT)

**Key Innovation**: Cross-song stem mixing — instead of mixing all 4 stems from the same song, we randomly pick each stem from a DIFFERENT song of the same genre. This exactly mimics the test data distribution where mashups combine stems from different songs.

# Cell 10 -- CrossSongMelDataset (Custom Dataset for CNN/CRNN)

This is the core data augmentation component of our pipeline. It defines a custom PyTorch `Dataset` class for CNN and CRNN models.

**What `__getitem__` does (step by step):**
1. **Genre Selection:** Picks a genre based on the current index.
2. **Cross-Song Stem Mixing:** For each of the 4 stems (vocals, drums, bass, other), it picks a **different random song** from the same genre. So vocals come from Song A, drums from Song B, bass from Song C, and other from Song D. This design closely mimics how the test mashups are created.
3. **Volume Augmentation:** Each stem is multiplied by a random gain between 0.7 and 1.3, varying the volume balance in every mix.
4. **Noise Injection (70% probability):** With 70% chance, a random ESC-50 noise clip is added at a random SNR (5 to 20 dB). This teaches the model to handle background noise.
5. **Normalization:** The final waveform is divided by its peak amplitude to normalize to [-1, 1].
6. **Mel Spectrogram Transformation:** The 1D waveform is converted to a 2D Mel Spectrogram using `MelSpectrogram` (n_fft=1024, hop_length=512, n_mels=64) and then scaled to decibels with `AmplitudeToDB`.
7. **Return:** `(mel_spec_db, label)` tuple.

**`DATASET_MULTIPLIER = 3`:** Since each call creates a new random mix, multiplying the length by 3 means the model sees 3000 unique audio combinations per epoch instead of 1000, significantly increasing data diversity.


# Cell 10 -- CrossSongMelDataset (Custom Dataset for CNN/CRNN)

This is the core data augmentation component of our pipeline. It defines a custom PyTorch `Dataset` class for CNN and CRNN models.

**What `__getitem__` does (step by step):**
1. **Genre Selection:** Picks a genre based on the current index.
2. **Cross-Song Stem Mixing:** For each of the 4 stems (vocals, drums, bass, other), it picks a **different random song** from the same genre. So vocals come from Song A, drums from Song B, bass from Song C, and other from Song D. This design closely mimics how the test mashups are created.
3. **Volume Augmentation:** Each stem is multiplied by a random gain between 0.7 and 1.3, varying the volume balance in every mix.
4. **Noise Injection (70% probability):** With 70% chance, a random ESC-50 noise clip is added at a random SNR (5 to 20 dB). This teaches the model to handle background noise.
5. **Normalization:** The final waveform is divided by its peak amplitude to normalize to [-1, 1].
6. **Mel Spectrogram Transformation:** The 1D waveform is converted to a 2D Mel Spectrogram using `MelSpectrogram` (n_fft=1024, hop_length=512, n_mels=64) and then scaled to decibels with `AmplitudeToDB`.
7. **Return:** `(mel_spec_db, label)` tuple.

**`DATASET_MULTIPLIER = 3`:** Since each call creates a new random mix, multiplying the length by 3 means the model sees 3000 unique audio combinations per epoch instead of 1000, significantly increasing data diversity.


In [10]:
# ============================================================
# Cell 10: CrossSongMelDataset (for CNN/CRNN — returns mel spectrogram)
# THIS IS THE KEY DATASET CLASS
# 3x dataset multiplication: each epoch sees 3000 unique cross-song mixes
# ============================================================

DATASET_MULTIPLIER = 3  # 3x more samples per epoch for more cross-song diversity

class CrossSongMelDataset(Dataset):
    """Cross-song stem mixing: picks each stem from a DIFFERENT song of the same genre.
    Dataset is multiplied 3x — since each __getitem__ randomly picks stems,
    every access generates a unique mixture even for repeated entries."""
    def __init__(self, stems_dir, noise_dir, genres, duration=DURATION, sample_rate=SAMPLE_RATE):
        self.sample_rate = sample_rate
        self.num_samples = sample_rate * duration
        self.stem_names = ['vocals.wav', 'drums.wav', 'bass.wav', 'other.wav']
        self.genre_songs = {}
        self.all_samples = []
        for genre in genres:
            genre_path = os.path.join(stems_dir, genre)
            if os.path.exists(genre_path):
                song_paths = [os.path.join(genre_path, s) for s in os.listdir(genre_path)]
                self.genre_songs[genre] = song_paths
                for _ in song_paths:
                    self.all_samples.append(genre)
        # 3x multiplication — each repeated entry still generates a unique random mix
        base_len = len(self.all_samples)
        self.all_samples = self.all_samples * DATASET_MULTIPLIER
        self.noise_files = []
        if noise_dir and os.path.exists(noise_dir):
            self.noise_files = [os.path.join(noise_dir, f) for f in os.listdir(noise_dir) if f.endswith('.wav')]
        self.mel_transform = T.MelSpectrogram(sample_rate=sample_rate, n_fft=1024, hop_length=512, n_mels=64)
        self.amplitude_to_db = T.AmplitudeToDB()
        print(f"CrossSongMelDataset: {len(self.all_samples)} samples ({base_len} x {DATASET_MULTIPLIER}), {len(self.noise_files)} noise files")

    def __len__(self):
        return len(self.all_samples)

    def _load_stem(self, song_path, stem_name):
        stem_path = os.path.join(song_path, stem_name)
        if not os.path.exists(stem_path):
            return torch.zeros(1, self.num_samples)
        waveform, sr = torchaudio.load(stem_path)
        if sr != self.sample_rate:
            waveform = T.Resample(sr, self.sample_rate)(waveform)
        waveform = torch.mean(waveform, dim=0, keepdim=True)
        total_len = waveform.shape[1]
        if total_len > self.num_samples:
            start = random.randint(0, total_len - self.num_samples)
            waveform = waveform[:, start:start + self.num_samples]
        elif total_len < self.num_samples:
            waveform = torch.nn.functional.pad(waveform, (0, self.num_samples - total_len))
        return waveform

    def __getitem__(self, idx):
        genre = self.all_samples[idx]
        label = genre_to_idx[genre]
        songs_in_genre = self.genre_songs[genre]
        mixed = torch.zeros(1, self.num_samples)
        for stem_name in self.stem_names:
            random_song = random.choice(songs_in_genre)
            stem_audio = self._load_stem(random_song, stem_name)
            mixed += stem_audio * random.uniform(0.7, 1.3)
        if self.noise_files and random.random() < 0.7:
            noise_wf, sr = torchaudio.load(random.choice(self.noise_files))
            if sr != self.sample_rate:
                noise_wf = T.Resample(sr, self.sample_rate)(noise_wf)
            noise_wf = torch.mean(noise_wf, dim=0, keepdim=True)
            if noise_wf.shape[1] < self.num_samples:
                noise_wf = torch.nn.functional.pad(noise_wf, (0, self.num_samples - noise_wf.shape[1]))
            else:
                noise_wf = noise_wf[:, :self.num_samples]
            signal_power = torch.mean(mixed ** 2) + 1e-10
            noise_power = torch.mean(noise_wf ** 2) + 1e-10
            snr_db = random.uniform(5, 20)
            scale = torch.sqrt(signal_power / (10 ** (snr_db / 10) * noise_power))
            mixed = mixed + scale * noise_wf
        mixed = mixed / (torch.max(torch.abs(mixed)) + 1e-8)
        mel_spec = self.mel_transform(mixed)
        mel_spec_db = self.amplitude_to_db(mel_spec)
        return mel_spec_db, label

print("CrossSongMelDataset class defined (with 3x multiplier).")

CrossSongMelDataset class defined (with 3x multiplier).


# Cell 12 -- DataLoader Setup (Train/Validation Splits)

Here we create PyTorch DataLoaders, which handle batching and shuffling during training.

**Key points:**
- Data is split 85% training / 15% validation.
- **Mel DataLoaders** (for CNN/CRNN): `batch_size=16`.
- **HuBERT DataLoaders**: `batch_size=8` (smaller because HuBERT processes raw waveforms of 160,000 samples through a large Transformer, requiring significantly more GPU memory).
- Training loaders use `shuffle=True`; validation loaders use `shuffle=False` for consistent evaluation.


# Cell 12 -- DataLoader Setup (Train/Validation Splits)

Here we create PyTorch DataLoaders, which handle batching and shuffling during training.

**Key points:**
- Data is split 85% training / 15% validation.
- **Mel DataLoaders** (for CNN/CRNN): `batch_size=16`.
- **HuBERT DataLoaders**: `batch_size=8` (smaller because HuBERT processes raw waveforms of 160,000 samples through a large Transformer, requiring significantly more GPU memory).
- Training loaders use `shuffle=True`; validation loaders use `shuffle=False` for consistent evaluation.


In [12]:
# ============================================================
# Cell 12: Build Data Loaders (GPU required from here)
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

mel_dataset = CrossSongMelDataset(STEMS_DIR, NOISE_DIR, genres)
mel_train_size = int(0.85 * len(mel_dataset))
mel_val_size = len(mel_dataset) - mel_train_size
mel_train_data, mel_val_data = random_split(mel_dataset, [mel_train_size, mel_val_size], generator=torch.Generator().manual_seed(SEED))

# Reduced batch sizes for 10s audio (2x longer = 2x more memory)
mel_train_loader = DataLoader(mel_train_data, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
mel_val_loader = DataLoader(mel_val_data, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

hubert_dataset = CrossSongHubertDataset(STEMS_DIR, NOISE_DIR, genres)
hub_train_size = int(0.85 * len(hubert_dataset))
hub_val_size = len(hubert_dataset) - hub_train_size
hub_train_data, hub_val_data = random_split(hubert_dataset, [hub_train_size, hub_val_size], generator=torch.Generator().manual_seed(SEED))

hub_train_loader = DataLoader(hub_train_data, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
hub_val_loader = DataLoader(hub_val_data, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nMel Dataset   - Train: {mel_train_size}, Val: {mel_val_size}")
print(f"HuBERT Dataset - Train: {hub_train_size}, Val: {hub_val_size}")
print(f"Duration: {DURATION}s | Mel batch: 16 | HuBERT batch: 8")

Device: cuda
CrossSongMelDataset: 3000 samples (1000 x 3), 2000 noise files
CrossSongHubertDataset: 3000 samples (1000 x 3)

Mel Dataset   - Train: 2550, Val: 450
HuBERT Dataset - Train: 2550, Val: 450
Duration: 10s | Mel batch: 16 | HuBERT batch: 8


# Cell 13 -- GenreCNN Model Architecture (Milestone 3)

This cell defines our CNN model for Mel Spectrogram classification.

**Architecture:**

**Feature Extractor (3 Convolutional Blocks):**
- **Block 1:** `Conv2d(1, 32, 3x3, padding=1)` -> `BatchNorm2d(32)` -> `ReLU` -> `MaxPool2d(2)`
- **Block 2:** `Conv2d(32, 64, ...)` -> `BatchNorm2d(64)` -> `ReLU` -> `MaxPool2d(2)`
- **Block 3:** `Conv2d(64, 128, ...)` -> `BatchNorm2d(128)` -> `ReLU` -> `AdaptiveAvgPool2d(1,1)`

**Classifier:** `Flatten` -> `Dropout(0.3)` -> `Linear(128, 10)`

**Design rationale:**
- **1 input channel:** Mel Spectrogram is like a grayscale image (single channel).
- **Channels increase 32 -> 64 -> 128:** Lower layers learn simple patterns (edges, basic frequencies). Higher layers combine them into complex genre-specific patterns, requiring more channels. This follows standard VGG/ResNet design principles.
- **BatchNorm2d:** Normalizes activations across the mini-batch: `(x - mean) / sqrt(var + eps)`, then applies learnable scale and shift. It stabilizes training, allows higher learning rates, and provides mild regularization. During `eval()` mode, it uses running mean/variance computed during training.
- **AdaptiveAvgPool2d(1,1):** Global Average Pooling -- reduces ANY feature map size to (1,1). Unlike MaxPool2d (which halves dimensions), this makes the model **input-size independent**, so different audio lengths are handled without hardcoded dimensions in the Linear layer.
- **Dropout(0.3):** Randomly zeroes 30% of neuron activations during training to prevent overfitting.


# Cell 13 -- GenreCNN Model Architecture (Milestone 3)

This cell defines our CNN model for Mel Spectrogram classification.

**Architecture:**

**Feature Extractor (3 Convolutional Blocks):**
- **Block 1:** `Conv2d(1, 32, 3x3, padding=1)` -> `BatchNorm2d(32)` -> `ReLU` -> `MaxPool2d(2)`
- **Block 2:** `Conv2d(32, 64, ...)` -> `BatchNorm2d(64)` -> `ReLU` -> `MaxPool2d(2)`
- **Block 3:** `Conv2d(64, 128, ...)` -> `BatchNorm2d(128)` -> `ReLU` -> `AdaptiveAvgPool2d(1,1)`

**Classifier:** `Flatten` -> `Dropout(0.3)` -> `Linear(128, 10)`

**Design rationale:**
- **1 input channel:** Mel Spectrogram is like a grayscale image (single channel).
- **Channels increase 32 -> 64 -> 128:** Lower layers learn simple patterns (edges, basic frequencies). Higher layers combine them into complex genre-specific patterns, requiring more channels. This follows standard VGG/ResNet design principles.
- **BatchNorm2d:** Normalizes activations across the mini-batch: `(x - mean) / sqrt(var + eps)`, then applies learnable scale and shift. It stabilizes training, allows higher learning rates, and provides mild regularization. During `eval()` mode, it uses running mean/variance computed during training.
- **AdaptiveAvgPool2d(1,1):** Global Average Pooling -- reduces ANY feature map size to (1,1). Unlike MaxPool2d (which halves dimensions), this makes the model **input-size independent**, so different audio lengths are handled without hardcoded dimensions in the Linear layer.
- **Dropout(0.3):** Randomly zeroes 30% of neuron activations during training to prevent overfitting.


In [13]:
# ============================================================
# Cell 13: CNN Model Definition (Milestone 3 — from scratch)
# ============================================================

class GenreCNN(nn.Module):
    """
    Simple CNN for genre classification from mel spectrograms.
    Uses BatchNorm, AdaptiveAvgPool (no hardcoded dims), and Dropout.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super(GenreCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),  # Global average pooling
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn_model = GenreCNN().to(device)
print(f"GenreCNN parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")
print(cnn_model)

GenreCNN parameters: 94,410
GenreCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.3, inplace=False)
    (2): Linear(in_features=128, out_features=10, bias=True)
  )


# Cell 14 -- CNN Training Loop + WandB Logging (Milestone 3)

This cell trains the CNN model. Here is how the training loop works:

1. `model.train()` -- Enables Dropout and sets BatchNorm to use current batch statistics.
2. Data is moved to GPU with `.to(device)`.
3. `optimizer.zero_grad()` -- Clears gradients from the previous iteration. PyTorch accumulates gradients by default, so without this step, gradients from different batches would add up and produce incorrect weight updates.
4. `torch.amp.autocast('cuda')` -- Enables Mixed Precision training (AMP). Runs computations in FP16 instead of FP32 for speed and memory savings.
5. `outputs = model(inputs)` -- Forward pass.
6. `loss = criterion(outputs, labels)` -- Computes CrossEntropyLoss with `label_smoothing=0.1`.
7. `scaler.scale(loss).backward()` -- Backpropagation. Computes gradients via chain rule. GradScaler scales the loss to prevent FP16 gradient underflow (very small gradients becoming zero). Note: `backward()` only computes and stores gradients, it does NOT update weights.
8. `scaler.step(optimizer)` -- **This is where weights are actually updated** (gradient descent). For Adam, the update rule adapts the learning rate per-parameter.
9. `scaler.update()` -- Adjusts the scale factor for the next iteration.
10. `scheduler.step()` -- Updates the learning rate. This is called **outside the batch loop** but inside the epoch loop (once per epoch). If placed inside the batch loop, LR would decay to zero within the first few batches.

**Validation:** Uses both `model.eval()` (disables Dropout, freezes BatchNorm) and `torch.no_grad()` (stops gradient tracking, saves memory). Both are needed together.

**Optimizer:** `Adam` with `lr=1e-3`. **Scheduler:** `CosineAnnealingLR` smoothly decays LR from initial value to near zero over `T_max` epochs.


# Cell 14 -- CNN Training Loop + WandB Logging (Milestone 3)

This cell trains the CNN model. Here is how the training loop works:

1. `model.train()` -- Enables Dropout and sets BatchNorm to use current batch statistics.
2. Data is moved to GPU with `.to(device)`.
3. `optimizer.zero_grad()` -- Clears gradients from the previous iteration. PyTorch accumulates gradients by default, so without this step, gradients from different batches would add up and produce incorrect weight updates.
4. `torch.amp.autocast('cuda')` -- Enables Mixed Precision training (AMP). Runs computations in FP16 instead of FP32 for speed and memory savings.
5. `outputs = model(inputs)` -- Forward pass.
6. `loss = criterion(outputs, labels)` -- Computes CrossEntropyLoss with `label_smoothing=0.1`.
7. `scaler.scale(loss).backward()` -- Backpropagation. Computes gradients via chain rule. GradScaler scales the loss to prevent FP16 gradient underflow (very small gradients becoming zero). Note: `backward()` only computes and stores gradients, it does NOT update weights.
8. `scaler.step(optimizer)` -- **This is where weights are actually updated** (gradient descent). For Adam, the update rule adapts the learning rate per-parameter.
9. `scaler.update()` -- Adjusts the scale factor for the next iteration.
10. `scheduler.step()` -- Updates the learning rate. This is called **outside the batch loop** but inside the epoch loop (once per epoch). If placed inside the batch loop, LR would decay to zero within the first few batches.

**Validation:** Uses both `model.eval()` (disables Dropout, freezes BatchNorm) and `torch.no_grad()` (stops gradient tracking, saves memory). Both are needed together.

**Optimizer:** `Adam` with `lr=1e-3`. **Scheduler:** `CosineAnnealingLR` smoothly decays LR from initial value to near zero over `T_max` epochs.


In [14]:
# ============================================================
# Cell 14: Train CNN + WandB (Milestone 3)
# ============================================================

wandb.init(project="22f1001611-t12026", name="cnn-from-scratch")

cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=1e-3)
cnn_scheduler = optim.lr_scheduler.CosineAnnealingLR(cnn_optimizer, T_max=10)
cnn_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
cnn_scaler = torch.amp.GradScaler('cuda')

CNN_EPOCHS = 10
best_cnn_f1 = 0.0

print(f"Training CNN for {CNN_EPOCHS} epochs on {device}...")
for epoch in range(CNN_EPOCHS):
    cnn_model.train()
    train_loss, correct, total = 0.0, 0, 0

    for inputs, labels in tqdm(mel_train_loader, desc=f"CNN Epoch {epoch+1}/{CNN_EPOCHS}"):
        inputs, labels = inputs.to(device), labels.to(device)
        cnn_optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = cnn_model(inputs)
            loss = cnn_criterion(outputs, labels)

        cnn_scaler.scale(loss).backward()
        cnn_scaler.step(cnn_optimizer)
        cnn_scaler.update()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total
    cnn_scheduler.step()

    # Validation
    cnn_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in mel_val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = cnn_model(inputs)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, average='macro')

    print(f"  Train Loss: {train_loss/len(mel_train_loader):.4f} | "
          f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss / len(mel_train_loader),
        "train_acc": train_acc,
        "val_acc": val_acc,
        "val_macro_f1": val_f1,
        "lr": cnn_optimizer.param_groups[0]['lr']
    })

    if val_f1 > best_cnn_f1:
        best_cnn_f1 = val_f1
        torch.save(cnn_model.state_dict(), 'best_cnn.pth')
        print(f"  -> New best CNN saved (F1={val_f1:.4f})")

cnn_val_acc = val_acc
cnn_val_f1 = best_cnn_f1
wandb.finish()
print(f"\nCNN Training Complete. Best Val Macro F1: {best_cnn_f1:.4f}")

wandb: setting up run ozvqa6fo
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260309_094331-ozvqa6fo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run cnn-from-scratch
wandb: ⭐️ View project at https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026
wandb: 🚀 View run at https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026/runs/ozvqa6fo


Training CNN for 10 epochs on cuda...


CNN Epoch 1/10:   0%|          | 0/160 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytor

  Train Loss: 1.8903 | Train Acc: 0.3612 | Val Acc: 0.4178 | Val F1: 0.3820
  -> New best CNN saved (F1=0.3820)


CNN Epoch 2/10:   0%|          | 0/160 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytor

...

  Train Loss: 1.3557 | Train Acc: 0.6573 | Val Acc: 0.6667 | Val F1: 0.6499
  -> New best CNN saved (F1=0.6499)


CNN Epoch 10/10:   0%|          | 0/160 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pyto

  Train Loss: 1.3551 | Train Acc: 0.6588 | Val Acc: 0.7111 | Val F1: 0.6996
  -> New best CNN saved (F1=0.6996)


wandb: 
wandb: Run history:
wandb:        epoch ▁▂▃▃▄▅▆▆▇█
wandb:           lr █▇▇▆▅▃▂▂▁▁
wandb:    train_acc ▁▃▄▅▆▆▇▇██
wandb:   train_loss █▆▅▄▃▃▂▂▁▁
wandb:      val_acc ▂▁▃▃▆▆▆▇▇█
wandb: val_macro_f1 ▂▁▃▃▆▆▆▇▇█
wandb: 
wandb: Run summary:
wandb:        epoch 10
wandb:           lr 0
wandb:    train_acc 0.65882
wandb:   train_loss 1.35513
wandb:      val_acc 0.71111
wandb: val_macro_f1 0.69959
wandb: 
wandb: 🚀 View run cnn-from-scratch at: https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026/runs/ozvqa6fo
wandb: ⭐️ View project at: https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260309_094331-ozvqa6fo/logs



CNN Training Complete. Best Val Macro F1: 0.6996


# Cell 15 -- CNN Evaluation

Loads the best CNN checkpoint (`best_cnn.pth`) and evaluates it on the validation set, printing a per-class classification report showing Precision, Recall, and F1 Score for each genre.


# Cell 15 -- CNN Evaluation

Loads the best CNN checkpoint (`best_cnn.pth`) and evaluates it on the validation set, printing a per-class classification report showing Precision, Recall, and F1 Score for each genre.


In [15]:
# ============================================================
# Cell 15: CNN Evaluation
# ============================================================

# Load best CNN and get per-class report
cnn_model.load_state_dict(torch.load('best_cnn.pth'))
cnn_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in mel_val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = cnn_model(inputs)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("CNN Per-Class Classification Report:")
print(classification_report(all_labels, all_preds, target_names=genres))
print(f"Best CNN Val Macro F1: {best_cnn_f1:.4f}")

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

CNN Per-Class Classification Report:
              precision    recall  f1-score   support

       blues       0.56      0.50      0.53        46
   classical       0.88      0.98      0.93        45
     country       0.65      0.70      0.67        46
       disco       0.52      0.49      0.51        45
      hiphop       0.81      0.92      0.86        50
        jazz       0.94      0.76      0.84        45
       metal       0.81      0.86      0.84        44
         pop       0.60      0.57      0.58        44
      reggae       0.67      0.67      0.67        49
        rock       0.57      0.58      0.58        36

    accuracy                           0.71       450
   macro avg       0.70      0.70      0.70       450
weighted avg       0.70      0.71      0.70       450

Best CNN Val Macro F1: 0.6996


# Cell 16 -- Memory Cleanup after CNN

GPU memory is limited. After CNN training, we delete the optimizer, scheduler, and scaler, run Python garbage collection (`gc.collect()`), and clear CUDA cache (`torch.cuda.empty_cache()`) to free memory for the next model.


# Cell 16 -- Memory Cleanup after CNN

GPU memory is limited. After CNN training, we delete the optimizer, scheduler, and scaler, run Python garbage collection (`gc.collect()`), and clear CUDA cache (`torch.cuda.empty_cache()`) to free memory for the next model.


In [16]:
# ============================================================
# Cell 16: Memory Cleanup after CNN
# ============================================================
del cnn_optimizer, cnn_scheduler, cnn_scaler
gc.collect()
torch.cuda.empty_cache()
print("CNN memory cleaned up.")

CNN memory cleaned up.
